# 👤 Reconnaissance Faciale Interactive

## 🎯 Objectifs
- Détecter et reconnaître des visages en temps réel
- Interface pour uploader vos propres photos
- Activation de la caméra pour reconnaissance live
- Pipeline complet de détection et reconnaissance

---
*Session 05 - SupNum Nouakchott - Formation IA & Machine Learning*

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
import base64
import os
from pathlib import Path
import face_recognition
import pickle
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Bibliothèques chargées avec succès!")
print("📸 Reconnaissance faciale prête!")

## 📁 1. Configuration et Stockage des Visages

### 🗂️ Structure des données :
- **faces_database/** : Dossier pour stocker les photos de référence
- **encodings.pkl** : Fichier des encodages des visages connus

In [ ]:
# Création des dossiers nécessaires
faces_dir = Path("faces_database")
faces_dir.mkdir(exist_ok=True)

print(f"📁 Dossier créé: {faces_dir}")
print("💡 Placez vos photos de référence dans ce dossier")

# Variables globales
known_face_encodings = []
known_face_names = []
encodings_file = "face_encodings.pkl"

## 🔧 2. Fonctions de Détection et Reconnaissance

In [ ]:
def load_and_encode_faces():
    """Charge et encode tous les visages du dossier faces_database"""
    global known_face_encodings, known_face_names
    
    known_face_encodings = []
    known_face_names = []
    
    print("🔍 Chargement des visages de référence...")
    
    # Parcourir tous les fichiers image
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    
    for image_path in faces_dir.iterdir():
        if image_path.suffix.lower() in image_extensions:
            try:
                # Charger l'image
                image = face_recognition.load_image_file(str(image_path))
                
                # Trouver les encodages des visages
                face_encodings = face_recognition.face_encodings(image)
                
                if face_encodings:
                    # Prendre le premier visage trouvé
                    face_encoding = face_encodings[0]
                    
                    # Utiliser le nom du fichier (sans extension) comme nom
                    name = image_path.stem
                    
                    known_face_encodings.append(face_encoding)
                    known_face_names.append(name)
                    
                    print(f"✅ Visage encodé: {name}")
                else:
                    print(f"❌ Aucun visage détecté dans: {image_path.name}")
                    
            except Exception as e:
                print(f"❌ Erreur avec {image_path.name}: {e}")
    
    # Sauvegarder les encodages
    if known_face_encodings:
        with open(encodings_file, 'wb') as f:
            pickle.dump({
                'encodings': known_face_encodings,
                'names': known_face_names
            }, f)
        print(f"💾 {len(known_face_encodings)} visages sauvegardés")
    else:
        print("⚠️ Aucun visage trouvé. Ajoutez des photos dans faces_database/")

def load_encodings():
    """Charge les encodages depuis le fichier"""
    global known_face_encodings, known_face_names
    
    if os.path.exists(encodings_file):
        with open(encodings_file, 'rb') as f:
            data = pickle.load(f)
            known_face_encodings = data['encodings']
            known_face_names = data['names']
        print(f"📂 {len(known_face_encodings)} visages chargés depuis le cache")
    else:
        print("📁 Aucun cache trouvé, chargement depuis les fichiers...")
        load_and_encode_faces()

# Charger les encodages au démarrage
load_encodings()

## 📤 3. Interface d'Upload de Photos de Référence

In [ ]:
# Widget pour uploader des photos de référence
upload_reference = widgets.FileUpload(
    accept='image/*',
    multiple=True,
    description='📤 Photos référence'
)

name_input = widgets.Text(
    placeholder='Entrez le nom de la personne',
    description='👤 Nom:',
    style={'description_width': 'initial'}
)

save_button = widgets.Button(
    description='💾 Sauvegarder',
    button_style='success'
)

output_reference = widgets.Output()

def save_reference_faces(button):
    """Sauvegarde les photos de référence uploadées"""
    with output_reference:
        clear_output(wait=True)
        
        if not upload_reference.value:
            print("❌ Aucune image sélectionnée")
            return
            
        if not name_input.value.strip():
            print("❌ Veuillez entrer un nom")
            return
        
        name = name_input.value.strip()
        saved_count = 0
        
        for filename, file_info in upload_reference.value.items():
            try:
                # Sauvegarder l'image
                image_path = faces_dir / f"{name}_{saved_count}.jpg"
                
                with open(image_path, 'wb') as f:
                    f.write(file_info['content'])
                
                saved_count += 1
                print(f"✅ Image sauvegardée: {image_path.name}")
                
            except Exception as e:
                print(f"❌ Erreur: {e}")
        
        if saved_count > 0:
            print(f"🔄 Rechargement des encodages...")
            load_and_encode_faces()
            print("✅ Base de données mise à jour!")
        
        # Reset
        upload_reference.value = {}
        name_input.value = ""

save_button.on_click(save_reference_faces)

print("📤 Interface d'Upload de Photos de Référence")
print("=" * 50)
print("📝 Instructions:")
print("1. Sélectionnez une ou plusieurs photos de la personne")
print("2. Entrez le nom de la personne")
print("3. Cliquez sur 'Sauvegarder'")

display(widgets.VBox([
    upload_reference,
    name_input,
    save_button,
    output_reference
]))

## 🖼️ 4. Test de Reconnaissance sur Photo

In [ ]:
def recognize_faces_in_image(image_array):
    """Reconnaît les visages dans une image"""
    if len(known_face_encodings) == 0:
        return image_array, []
    
    # Convertir en RGB si nécessaire
    if len(image_array.shape) == 3 and image_array.shape[2] == 3:
        rgb_image = image_array
    else:
        rgb_image = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
    
    # Détecter les visages
    face_locations = face_recognition.face_locations(rgb_image)
    face_encodings = face_recognition.face_encodings(rgb_image, face_locations)
    
    results = []
    
    # Dessiner les rectangles et noms
    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
        # Comparer avec les visages connus
        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
        name = "Inconnu"
        confidence = 0
        
        # Calculer les distances
        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
        
        if len(face_distances) > 0:
            best_match_index = np.argmin(face_distances)
            if matches[best_match_index]:
                name = known_face_names[best_match_index]
                confidence = 1 - face_distances[best_match_index]
        
        results.append({
            'name': name,
            'confidence': confidence,
            'location': (top, right, bottom, left)
        })
        
        # Dessiner le rectangle
        color = (0, 255, 0) if name != "Inconnu" else (255, 0, 0)
        cv2.rectangle(rgb_image, (left, top), (right, bottom), color, 2)
        
        # Dessiner le nom et la confiance
        label = f"{name} ({confidence:.1%})" if name != "Inconnu" else name
        cv2.rectangle(rgb_image, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
        cv2.putText(rgb_image, label, (left + 6, bottom - 6), 
                   cv2.FONT_HERSHEY_DUPLEX, 0.6, (255, 255, 255), 1)
    
    return rgb_image, results

# Widget pour tester la reconnaissance
upload_test = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='📸 Test photo'
)

output_test = widgets.Output()

def test_recognition(change):
    """Test de reconnaissance sur photo uploadée"""
    with output_test:
        clear_output(wait=True)
        
        if not upload_test.value:
            return
        
        if len(known_face_encodings) == 0:
            print("❌ Aucun visage de référence. Ajoutez d'abord des photos!")
            return
        
        try:
            # Charger l'image
            uploaded_file = list(upload_test.value.values())[0]
            image_bytes = io.BytesIO(uploaded_file['content'])
            image = Image.open(image_bytes)
            image_array = np.array(image)
            
            # Reconnaissance
            result_image, results = recognize_faces_in_image(image_array)
            
            # Affichage
            fig, axes = plt.subplots(1, 2, figsize=(15, 6))
            
            # Image originale
            axes[0].imshow(image_array)
            axes[0].set_title('📸 Image Originale', fontweight='bold')
            axes[0].axis('off')
            
            # Image avec reconnaissance
            axes[1].imshow(result_image)
            axes[1].set_title('🎯 Reconnaissance Faciale', fontweight='bold')
            axes[1].axis('off')
            
            plt.tight_layout()
            plt.show()
            
            # Résultats textuels
            print("🎯 RÉSULTATS DE LA RECONNAISSANCE:")
            print("=" * 40)
            
            if results:
                for i, result in enumerate(results, 1):
                    name = result['name']
                    confidence = result['confidence']
                    
                    if name != "Inconnu":
                        print(f"👤 Visage {i}: {name} (Confiance: {confidence:.1%})")
                        if confidence > 0.8:
                            print("   ✅ Reconnaissance très fiable")
                        elif confidence > 0.6:
                            print("   ⚠️ Reconnaissance modérée")
                        else:
                            print("   ❓ Reconnaissance incertaine")
                    else:
                        print(f"❓ Visage {i}: Personne inconnue")
            else:
                print("❌ Aucun visage détecté dans l'image")
                
        except Exception as e:
            print(f"❌ Erreur: {e}")

upload_test.observe(test_recognition, names='value')

print("🖼️ Test de Reconnaissance sur Photo")
print("=" * 40)
print("📝 Uploadez une photo pour tester la reconnaissance")

display(upload_test, output_test)

## 📹 5. Reconnaissance en Temps Réel avec Webcam

In [ ]:
class WebcamRecognition:
    def __init__(self):
        self.cap = None
        self.is_running = False
        
    def start_camera(self):
        """Démarre la caméra"""
        try:
            self.cap = cv2.VideoCapture(0)
            if not self.cap.isOpened():
                print("❌ Impossible d'accéder à la caméra")
                return False
            
            # Configuration de la caméra
            self.cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            self.cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            self.cap.set(cv2.CAP_PROP_FPS, 30)
            
            print("✅ Caméra initialisée")
            return True
        except Exception as e:
            print(f"❌ Erreur caméra: {e}")
            return False
    
    def stop_camera(self):
        """Arrête la caméra"""
        if self.cap:
            self.cap.release()
            self.cap = None
        self.is_running = False
        cv2.destroyAllWindows()
        print("📹 Caméra arrêtée")
    
    def run_recognition(self):
        """Lance la reconnaissance en temps réel"""
        if len(known_face_encodings) == 0:
            print("❌ Aucun visage de référence!")
            print("💡 Ajoutez d'abord des photos de référence")
            return
        
        if not self.start_camera():
            return
        
        self.is_running = True
        print("🎥 Reconnaissance en cours... Appuyez sur 'q' pour quitter")
        
        # Optimisation: traiter 1 frame sur 4 pour la reconnaissance
        frame_count = 0
        
        try:
            while self.is_running:
                ret, frame = self.cap.read()
                if not ret:
                    break
                
                frame_count += 1
                
                # Reconnaissance tous les 4 frames pour optimiser
                if frame_count % 4 == 0:
                    # Redimensionner pour accélérer
                    small_frame = cv2.resize(frame, (0, 0), fx=0.25, fy=0.25)
                    rgb_small_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
                    
                    # Détecter les visages
                    face_locations = face_recognition.face_locations(rgb_small_frame)
                    face_encodings = face_recognition.face_encodings(rgb_small_frame, face_locations)
                    
                    # Reconnaissance
                    for (top, right, bottom, left), face_encoding in zip(face_locations, face_encodings):
                        matches = face_recognition.compare_faces(known_face_encodings, face_encoding)
                        name = "Inconnu"
                        confidence = 0
                        
                        face_distances = face_recognition.face_distance(known_face_encodings, face_encoding)
                        
                        if len(face_distances) > 0:
                            best_match_index = np.argmin(face_distances)
                            if matches[best_match_index]:
                                name = known_face_names[best_match_index]
                                confidence = 1 - face_distances[best_match_index]
                        
                        # Redimensionner les coordonnées
                        top *= 4
                        right *= 4
                        bottom *= 4
                        left *= 4
                        
                        # Dessiner sur l'image complète
                        color = (0, 255, 0) if name != "Inconnu" else (0, 0, 255)
                        cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
                        
                        label = f"{name} ({confidence:.1%})" if name != "Inconnu" else name
                        cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
                        cv2.putText(frame, label, (left + 6, bottom - 6), 
                                   cv2.FONT_HERSHEY_DUPLEX, 0.8, (255, 255, 255), 1)
                
                # Afficher le frame
                cv2.imshow('🎥 Reconnaissance Faciale Live', frame)
                
                # Quitter avec 'q'
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
                    
        except KeyboardInterrupt:
            print("\n⏹️ Arrêt demandé")
        except Exception as e:
            print(f"❌ Erreur: {e}")
        finally:
            self.stop_camera()

# Instance de reconnaissance webcam
webcam_recognition = WebcamRecognition()

# Boutons de contrôle
start_button = widgets.Button(
    description='🎥 Démarrer Webcam',
    button_style='success'
)

stop_button = widgets.Button(
    description='⏹️ Arrêter',
    button_style='danger'
)

webcam_output = widgets.Output()

def start_webcam(button):
    """Démarre la reconnaissance webcam"""
    with webcam_output:
        clear_output(wait=True)
        print("🎥 Démarrage de la reconnaissance en temps réel...")
        print("💡 Une fenêtre va s'ouvrir - appuyez sur 'q' pour quitter")
        
    # Lancer dans un thread séparé pour éviter de bloquer Jupyter
    import threading
    thread = threading.Thread(target=webcam_recognition.run_recognition)
    thread.daemon = True
    thread.start()

def stop_webcam(button):
    """Arrête la reconnaissance webcam"""
    with webcam_output:
        webcam_recognition.is_running = False
        webcam_recognition.stop_camera()
        print("⏹️ Reconnaissance arrêtée")

start_button.on_click(start_webcam)
stop_button.on_click(stop_webcam)

print("📹 Reconnaissance Faciale en Temps Réel")
print("=" * 45)
print("⚠️ Assurez-vous d'avoir une webcam connectée")
print("💡 La fenêtre de reconnaissance s'ouvrira dans une nouvelle fenêtre")

display(widgets.HBox([start_button, stop_button]), webcam_output)

## 📊 6. Statistiques et Gestion de la Base de Données

In [ ]:
def show_database_stats():
    """Affiche les statistiques de la base de données"""
    print("📊 STATISTIQUES DE LA BASE DE DONNÉES")
    print("=" * 45)
    
    if len(known_face_names) == 0:
        print("❌ Base de données vide")
        print("💡 Ajoutez des photos de référence pour commencer")
        return
    
    print(f"👥 Nombre de personnes: {len(known_face_names)}")
    print(f"🗂️ Dossier: {faces_dir}")
    
    # Compter les fichiers par personne
    person_counts = {}
    for image_path in faces_dir.iterdir():
        if image_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
            # Extraire le nom (avant le premier underscore)
            name_part = image_path.stem.split('_')[0]
            person_counts[name_part] = person_counts.get(name_part, 0) + 1
    
    print("\n👤 Personnes enregistrées:")
    for i, name in enumerate(known_face_names, 1):
        file_count = person_counts.get(name, 0)
        print(f"   {i}. {name} ({file_count} photo{'s' if file_count > 1 else ''})")
    
    # Graphique
    if len(known_face_names) > 0:
        plt.figure(figsize=(10, 6))
        names_unique = list(set(known_face_names))
        counts = [person_counts.get(name, 0) for name in names_unique]
        
        bars = plt.bar(names_unique, counts, color=sns.color_palette("husl", len(names_unique)))
        plt.title('📊 Nombre de Photos par Personne', fontweight='bold')
        plt.xlabel('Personnes')
        plt.ylabel('Nombre de photos')
        plt.xticks(rotation=45)
        
        # Ajouter les valeurs sur les barres
        for bar, count in zip(bars, counts):
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 0.1,
                     f'{count}', ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

# Bouton pour afficher les stats
stats_button = widgets.Button(
    description='📊 Voir Statistiques',
    button_style='info'
)

clear_db_button = widgets.Button(
    description='🗑️ Vider Base',
    button_style='warning'
)

stats_output = widgets.Output()

def show_stats(button):
    """Affiche les statistiques"""
    with stats_output:
        clear_output(wait=True)
        show_database_stats()

def clear_database(button):
    """Vide la base de données"""
    with stats_output:
        clear_output(wait=True)
        
        # Supprimer tous les fichiers
        for image_path in faces_dir.iterdir():
            if image_path.is_file():
                image_path.unlink()
        
        # Supprimer le cache
        if os.path.exists(encodings_file):
            os.remove(encodings_file)
        
        # Réinitialiser les variables
        global known_face_encodings, known_face_names
        known_face_encodings = []
        known_face_names = []
        
        print("🗑️ Base de données vidée")
        print("✅ Tous les fichiers supprimés")

stats_button.on_click(show_stats)
clear_db_button.on_click(clear_database)

display(widgets.HBox([stats_button, clear_db_button]), stats_output)

## 🎓 7. Guide d'Utilisation et Conseils

### 📝 Étapes pour utiliser ce notebook :

1. **📤 Ajouter des photos de référence** :
   - Uploadez des photos claires de chaque personne
   - Utilisez des photos avec un seul visage visible
   - Variez les angles et expressions

2. **🖼️ Tester sur des photos** :
   - Uploadez une photo pour voir la reconnaissance
   - Vérifiez la précision et la confiance

3. **📹 Reconnaissance en temps réel** :
   - Cliquez sur "Démarrer Webcam"
   - Positionnez-vous face à la caméra
   - Appuyez sur 'q' pour quitter

### 💡 Conseils pour de meilleurs résultats :

- **Photos de qualité** : Utilisez des images nettes et bien éclairées
- **Visages frontaux** : Les photos de face donnent de meilleurs résultats
- **Plusieurs angles** : Ajoutez plusieurs photos par personne
- **Bon éclairage** : Évitez les contre-jours et ombres fortes
- **Distance appropriée** : Le visage doit occuper une bonne partie de l'image

### ⚠️ Limitations :

- Fonctionne mieux avec des visages frontaux
- Sensible aux changements d'éclairage
- Peut confondre des personnes similaires
- Performance dépend de la qualité des photos de référence

---
**🎉 Amusez-vous avec la reconnaissance faciale !**